# Preprocessing Pipeline

**Tujuan**: Preprocessing teks ulasan e-commerce untuk real-time inference.

## Pipeline
1. `normalize_repetitive_chars` — normalisasi karakter berulang
2. `normalize_slang` — normalisasi slang (kamus alay)
3. `emoji_to_text` — konversi emoji ke teks
4. `tokenize` — tokenisasi NLTK `word_tokenize`
5. `remove_stopwords` — hapus stopwords
6. `pos_tag` — POS tagging rule-based
7. `stem` — stemming Sastrawi
8. `handle_negation` — penanganan negasi
9. `extract_emotion_features` — ekstraksi fitur emosi

In [25]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from collections import Counter
from scipy.sparse import hstack
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
import emoji
import pandas as pd
import re
import json
import time
from collections import OrderedDict
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Load Data

In [26]:
df = pd.read_csv("PRDECT-ID Dataset.csv")
df["text"] = df["Customer Review"].copy()
df["text-original"] = df["Customer Review"].copy()

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nSentiment distribution:\n{df['Sentiment'].value_counts()}")
print(f"\nEmotion distribution:\n{df['Emotion'].value_counts()}")

Total rows: 5400
Columns: ['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Customer Review', 'Sentiment', 'Emotion', 'text', 'text-original']

Sentiment distribution:
Sentiment
Negative    2821
Positive    2579
Name: count, dtype: int64

Emotion distribution:
Emotion
Happy      1770
Sadness    1202
Fear        920
Love        809
Anger       699
Name: count, dtype: int64


### Load Kamus Alay — Colloquial Indonesian Lexicon

Sumber: [nasalsabila/kamus-alay](https://github.com/nasalsabila/kamus-alay) — 15.397 entri slang → formal.

In [27]:
kamus_df = pd.read_csv("colloquial-indonesian-lexicon.csv")
kamus_df.head(3)

,slang,formal,In-dictionary,context,category1,category2,category3
0,woww,wow,1,wow,elongasi,0,0
1,aminn,amin,1,Selamat ulang tahun kakak tulus semoga panjang...,elongasi,0,0
2,met,selamat,1,Met hari netaas kak!? Wish you all the best @t...,abreviasi,0,0


In [28]:
# Filter hanya yang In-dictionary=1 (paling terpercaya)
kamus_valid = kamus_df[kamus_df["In-dictionary"] == 1]
print(f"Total entri: {len(kamus_df)}")
print(f"Entri valid (In-dictionary=1): {len(kamus_valid)}")

# Build dictionary: slang -> formal
# Prioritaskan formal yang paling pendek jika ada duplikat slang
slang_dict = {}
for _, row in kamus_valid.iterrows():
    slang = str(row["slang"]).strip().lower()
    formal = str(row["formal"]).strip().lower()
    if slang in slang_dict:
        if len(formal) < len(slang_dict[slang]):
            slang_dict[slang] = formal
    else:
        slang_dict[slang] = formal

print(f"Unique slang entries in dictionary: {len(slang_dict)}")

Total entri: 15006
Entri valid (In-dictionary=1): 13722
Unique slang entries in dictionary: 3451


### Preprocessing Functions

In [30]:
def normalize_repetitive_chars(text: str) -> str:
    # Vokal: a,i,u,e,o -> 1 char
    text = re.sub(r"(a)\1{2,}", "a", text)
    text = re.sub(r"(i)\1{2,}", "i", text)
    text = re.sub(r"(u)\1{2,}", "u", text)
    text = re.sub(r"(e)\1{2,}", "e", text)
    text = re.sub(r"(o)\1{2,}", "o", text)
    # Konsonan: 2 char max
    text = re.sub(r"([^aiueo])\1{2,}", r"\1", text)
    return text

def normalize_slang(text: str, slang_dict: dict) -> str:
    words = text.split()
    normalized = []
    for w in words:
        w_lower = w.lower()
        if w_lower in slang_dict:
            normalized.append(slang_dict[w_lower])
        elif w_lower.rstrip(".,!?;:") in slang_dict:
            punct = w[len(w_lower.rstrip(".,!?;:")):]
            normalized.append(slang_dict[w_lower.rstrip(".,!?;:")] + punct)
        else:
            normalized.append(w)
    return " ".join(normalized)


def tokenize(text: str) -> list:
    return word_tokenize(text)


def remove_stopwords(tokens: list) -> list:
    stop_words = set(stopwords.words('indonesian'))
    return [t for t in tokens if t.lower() not in stop_words]


def pos_tag(tokens: list) -> list:
    import re
    konjungsi = {'dan', 'atau', 'tetapi', 'namun', 'sedangkan', 'serta',
                 'karena', 'sehingga', 'maka', 'lalu', 'kemudian',
                 'setelah', 'sebelum', 'ketika', 'sementara', 'walaupun',
                 'meskipun', 'jika', 'kalau', 'apabila', 'bahwa'}
    preposisi = {'di', 'ke', 'dari', 'pada', 'dengan', 'untuk', 'bagi',
                 'oleh', 'tentang', 'seperti', 'sebagai', 'tanpa', 'dalam',
                 'antara', 'menurut', 'sampai', 'hingga'}
    hasil = []
    for token in tokens:
        t = token.lower()
        if re.match(r'^[.,!?;:()\[\]{}"\'\-]$', token):
            hasil.append((token, 'PUNCT'))
        elif re.match(r'^[0-9,.\-]+$', token):
            hasil.append((token, 'NUM'))
        elif t in konjungsi:
            hasil.append((token, 'CONJ'))
        elif t in preposisi:
            hasil.append((token, 'ADP'))
        elif re.match(r'^(me|men|meng|meny|mem|di|ber|bel|ter|per)', t):
            hasil.append((token, 'VERB'))
        elif re.match(r'^(pe|pen|pem|peng|ke)', t) or t.endswith('an') or t.endswith('kan'):
            hasil.append((token, 'NOUN'))
        elif t.endswith('i'):
            hasil.append((token, 'VERB'))
        else:
            hasil.append((token, 'NOUN'))
    return hasil


def stem(tokens: list, stemmer) -> list:
    return [stemmer.stem(t) for t in tokens]

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def emoji_to_text(text: str) -> str:
    return emoji.demojize(text, language='id')


def handle_negation(tokens: list, neg_words: set = None) -> list:
    if neg_words is None:
        neg_words = {
            "tidak", "bukan", "belum", "tak", "ngga",
            "gak", "ga", "tdk", "enggak", "nggak",
            "kagak", "ndak", "ngg"
        }
    result = []
    negate = False
    for w in tokens:
        w_clean = w.lower().strip(".,!?")
        if w_clean in neg_words:
            result.append(w)
            negate = True
        elif negate:
            result.append("NEG_" + w)
            negate = False
        else:
            result.append(w)
    return result
    
def extract_emotion_features(series: pd.Series) -> pd.DataFrame:
    import itertools
    features = pd.DataFrame(index=series.index)
    features["n_exclamation"] = series.apply(lambda x: x.count("!"))
    features["n_question"] = series.apply(lambda x: x.count("?"))
    features["n_allcaps"] = series.apply(
        lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 2)
    )
    features["n_ellipsis"] = series.apply(lambda x: x.count(".."))
    features["max_char_repeat"] = series.apply(
        lambda x: max(
            (len(list(g)) for _, g in itertools.groupby(str(x).lower())),
            default=0,
        )
    )
    return features
def full_pipeline(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return text
    t = text
    t = normalize_repetitive_chars(t)
    t = normalize_slang(t, slang_dict)
    t = emoji_to_text(t)
    tokens = tokenize(t)
    tokens = remove_stopwords(tokens)
    pos_tags = pos_tag(tokens)
    tokens = stem(tokens, stemmer)
    tokens = handle_negation(tokens)
    return " ".join(tokens)

In [31]:
X_raw = df["Customer Review"].fillna("")
y_emo = df["Emotion"]
y_sen = df["Sentiment"]

max_features=1500

## Train & Test Split Fit

In [32]:
X_train_text, X_test_text, y_train_emo, y_test_emo, y_train_sen, y_test_sen = train_test_split(
    X_raw, y_emo, y_sen, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=max_features)

# TfidfVectorizer only
X_train_vec = vectorizer.fit_transform(X_train_text.apply(full_pipeline))
X_test_vec = vectorizer.transform(X_test_text.apply(full_pipeline))

emotion_train = extract_emotion_features(X_train_text)
emotion_test = extract_emotion_features(X_test_text)

# TfidfVectorizer + Emotion Features
X_train_comb = hstack([X_train_vec, emotion_train.values])
X_test_comb = hstack([X_test_vec, emotion_test.values])

# TfidfVectorizer + Emotion Features + Sentiment
sent_train = (y_train_sen == "Positive").astype(int).values.reshape(-1, 1)
sent_test = (y_test_sen == "Positive").astype(int).values.reshape(-1, 1)
X_train_comb_sent = hstack([X_train_vec, emotion_train.values, sent_train])
X_test_comb_sent = hstack([X_test_vec, emotion_test.values, sent_test])

In [33]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

f1_emos_a, f1_sens_a = [], []
f1_emos_b, f1_sens_b = [], []

for train_idx, test_idx in skf.split(X_raw, y_emo):
    X_tr, X_te = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
    y_emo_tr, y_emo_te = y_emo.iloc[train_idx], y_emo.iloc[test_idx]
    y_sen_tr, y_sen_te = y_sen.iloc[train_idx], y_sen.iloc[test_idx]

    # TfidfVectorizer only
    vec_a = TfidfVectorizer(max_features=max_features)
    X_tr_a = vec_a.fit_transform(X_tr.apply(full_pipeline))
    X_te_a = vec_a.transform(X_te.apply(full_pipeline))

    # TfidfVectorizer + Emotion Features
    emo_tr = extract_emotion_features(X_tr)
    emo_te = extract_emotion_features(X_te)
    X_tr_b = hstack([X_tr_a, emo_tr.values])
    X_te_b = hstack([X_te_a, emo_te.values])

    # TfidfVectorizer + Emotion Features + Sentiment
    sent_tr = (y_sen_tr == "Positive").astype(int).values.reshape(-1, 1)
    sent_te = (y_sen_te == "Positive").astype(int).values.reshape(-1, 1)
    X_tr_c = hstack([X_tr_a, emo_tr.values, sent_tr])
    X_te_c = hstack([X_te_a, emo_te.values, sent_te])

## Full Dataset Fit

In [34]:
# TfidfVectorizer only
X_vec = vectorizer.fit_transform(X_raw)

emotion_feature = extract_emotion_features(df["Customer Review"])

# TfidfVectorizer + Emotion Features
X_vec_comb = hstack([X_vec, emotion_feature.values])

# TfidfVectorizer + Emotion Features + Sentiment
sentiment_feature = (df["Sentiment"] == "Positive").astype(int).values.reshape(-1, 1)
X_vec_comb_sent = hstack([X_vec, emotion_feature.values, sentiment_feature])